In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PowerTransformer, StandardScaler
import os 
import json
import numpy as np

In [ ]:
ace=pd.read_pickle('/mnt/disks/data/ACE/ace_full.pkl')
dscovr=pd.read_pickle('/mnt/disks/data/DSCOVR/dscovr_full.pkl')

In [ ]:
full_data=pd.concat([ace,dscovr],axis=0,ignore_index=True)

In [ ]:
pt = PowerTransformer(method='yeo-johnson', standardize=True)

In [ ]:
full_data_scaler=pt.fit(full_data)

In [ ]:
def save_scaler(scaler, scaler_dir, filename):
    """
    Saves a PowerTransfomer(method='yeo-johnson', standardize=True) object to a JSON file.

    Args:
        scaler (PowerTransfomer): The scaler object to save.
        filename (str): Name of the JSON file to save the scaler.
    """
    pt_scaler_dict = {
        "params": scaler.get_params(),
        "lambdas": scaler.lambdas_.tolist(),
        "n_features": scaler.n_features_in_,
        "feature_names": scaler.feature_names_in_.tolist(),
    }
    std_scaler_dict = {
        "mean": scaler._scaler.mean_.tolist(),
        "scale": scaler._scaler.scale_.tolist(),
        "var": scaler._scaler.var_.tolist(),
    }
    with open(os.path.join(scaler_dir, 'pt_'+filename), "w") as f:
        json.dump(pt_scaler_dict, f)
        
    with open(os.path.join(scaler_dir, 'std_'+filename), "w") as f:
        json.dump(std_scaler_dict, f)

In [ ]:
def load_pt_scaler(scaler_dir, filename):
    """
    Loads a PowerTransformer and associated StandardScaler object from JSON files.

    Args:
        filename (str): Name of the JSON file containing the scaler.

    Returns:
        PowerTransformer: The loaded scaler object.
    """
    with open(os.path.join(scaler_dir, 'pt_'+filename), "r") as f:
        pt_scaler_dict = json.load(f)
    pt_scaler = PowerTransformer(method='yeo-johnson', standardize=False)
    for key, value in pt_scaler_dict["params"].items():
        pt_scaler.set_params(**{key: value})
    pt_scaler.lambdas_ = np.array(pt_scaler_dict["lambdas"])
    pt_scaler.n_features_in_ = pt_scaler_dict["n_features"]
    pt_scaler.feature_names_in_ = np.array(pt_scaler_dict["feature_names"])
    
    with open(os.path.join(scaler_dir, 'std_'+filename), "r") as f:
        std_scaler_dict = json.load(f)
    std_scaler = StandardScaler()
    std_scaler.mean_ = np.array(std_scaler_dict["mean"])
    std_scaler.scale_ = np.array(std_scaler_dict["scale"])
    std_scaler.var_ = np.array(std_scaler_dict["var"])
            
    return pt_scaler, std_scaler

In [ ]:
save_scaler(pt, '.', 'scaler.json')

#apply the scaler to the data - note the double step to include the standardization from the original fit
pt_tst, std_tst = load_pt_scaler('.', 'scaler.json')
pt_tst._scaler = std_tst
pt_tst.set_output(transform="pandas")

In [ ]:
full_data_fitted = pt_tst.transform(full_data)

In [ ]:
# plot the transformed data

for col in full_data_fitted.columns:
    plt.hist(full_data_fitted[col], bins=100)
    plt.title(col)
    plt.show()

In [ ]:
full_data_inv = pt_tst.inverse_transform(full_data_fitted)

In [ ]:
# plot distributions of all collumns in full_data_scaler
full_data_scaler_df=pd.DataFrame(full_data_inv,columns=full_data.columns)

for col in full_data_scaler_df.columns:
    plt.figure()
    full_data_scaler_df[col].plot.hist(bins=100)
    full_data[col].plot.hist(bins=100,alpha=0.5)
    plt.title(col)
    plt.show()